In [ ]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict
from langgraph.checkpoint.memory import InMemorySaver
import time

In [ ]:
class CrashState(TypedDict):
    input:str
    step1:str
    step2:str
    step3:str

In [ ]:
from matplotlib.pyplot import step


def step_1(state:CrashState):
    print("step 1 executed")
    return {'step1':'done','input':state['input']}
    
def step_2(state:CrashState):
    print("Step 2 hanging.... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(30)
    return {"step2":"done"}
def step_3(state:CrashState):
    print('Step 3 execute')
    return {'step3':'done'}

In [ ]:
checkpointer=InMemorySaver()
builder=StateGraph(CrashState)
builder.add_node('step_1',step_1)
builder.add_node('step_2',step_2)
builder.add_node('step_3',step_3)
builder.add_edge(START,'step_1')
builder.add_edge('step_1','step_2')
builder.add_edge('step_2','step_3')
builder.add_edge('step_3',END)
workflow=builder.compile(checkpointer=checkpointer)
builder.compile()

In [ ]:
try:
    print('running graph: Please manually interrupt during step 2')
    workflow.invoke(
        {'input': "start"},
        config={'configurable': {'thread_id': 'thread-1'}},  # fixed: 'configurable'
    )
except KeyboardInterrupt:
    print("kernel manually interrupted")

In [ ]:
list(workflow.get_state({'configurable':{'thread_id':'thread-1'}}))